# ⚡ YOLOv11n — Esophageal Cancer Detection (Lightweight)
### 150 Epochs | Loss Curves | Confusion Matrix | Metrics Table | Model Comparison | best.pt
YOLOv11**n** (nano, ~3.0M params) — the lightest, fastest model in this study. Same COCO dataset / output format as your other notebooks so results are directly comparable.

> Want more accuracy for a small time cost? Change `MODEL_VARIANT` in Cell 3 to `'yolo11s.pt'` (small, ~9.4M params, still fast).

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Install
# ═══════════════════════════════════════════════════════════
import os
os.system('pip install ultralytics -q')
print('✅ Done')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Imports
# ═══════════════════════════════════════════════════════════
import json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
from PIL import Image
from collections import defaultdict
import torch
from ultralytics import YOLO
warnings.filterwarnings('ignore')
print(f'PyTorch    : {torch.__version__}')
print(f'GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — enable GPU!"}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — CONFIG  ← only change COCO_ROOT / MODEL_VARIANT if needed
# ═══════════════════════════════════════════════════════════
COCO_ROOT     = '/kaggle/input/datasets/rugvedkulkarni2509/esophageal-balanced-coco/coco_balanced'
ANNOTS_DIR    = COCO_ROOT + '/annotations'
YOLO_ROOT     = '/kaggle/working/yolo_data'          # converted YOLO-format dataset lives here
OUTPUT_DIR    = '/kaggle/working/yolo11n_output'
MODEL_VARIANT = 'yolo11n.pt'    # lightweight nano — swap to 'yolo11s.pt' for a bit more accuracy
EPOCHS        = 150
IMG_SIZE      = 640
BATCH_SIZE    = 16              # YOLO is light enough for a bigger batch than the R-CNN models
DEVICE        = 0 if torch.cuda.is_available() else 'cpu'
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(f'{ANNOTS_DIR}/instances_train.json') as f:
    _d = json.load(f)
CLASS_NAMES = [c['name'] for c in sorted(_d['categories'], key=lambda x: x['id'])]
NUM_CLASSES = len(CLASS_NAMES)

print('✅ Config OK')
print(f'   COCO_ROOT   : {COCO_ROOT}')
print(f'   Classes     : {CLASS_NAMES}')
print(f'   NUM_CLASSES : {NUM_CLASSES}')
print(f'   Model       : {MODEL_VARIANT}')
print(f'   Epochs      : {EPOCHS}')
print(f'   Img size    : {IMG_SIZE}')
print(f'   Batch size  : {BATCH_SIZE}')
print(f'   Device      : {DEVICE}')

print()
for sp in ['train', 'val', 'test']:
    jf   = Path(ANNOTS_DIR)/f'instances_{sp}.json'
    idir = Path(COCO_ROOT)/sp
    with open(jf) as f: dd = json.load(f)
    nimgs = len(dd['images']); nanns = len(dd['annotations'])
    nfiles = len(list(idir.glob('*.jpg')))+len(list(idir.glob('*.png')))
    print(f'  ✅ {sp:5s} → {nimgs} images | {nanns} annotations | {nfiles} files')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Convert COCO-format dataset → YOLO format
# ═══════════════════════════════════════════════════════════
def convert_coco_to_yolo(coco_root, annots_dir, out_root, class_names, splits=('train', 'val', 'test')):
    cats_sorted = sorted(json.load(open(f'{annots_dir}/instances_train.json'))['categories'], key=lambda x: x['id'])
    cat_map = {c['id']: i for i, c in enumerate(cats_sorted)}  # 0-indexed for YOLO

    for sp in splits:
        img_dir  = Path(coco_root) / sp
        ann_file = Path(annots_dir) / f'instances_{sp}.json'
        with open(ann_file) as f:
            dd = json.load(f)
        abi = defaultdict(list)
        for a in dd['annotations']:
            abi[a['image_id']].append(a)

        out_img_dir = Path(out_root) / 'images' / sp
        out_lbl_dir = Path(out_root) / 'labels' / sp
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_lbl_dir.mkdir(parents=True, exist_ok=True)

        n_written = 0
        for img in dd['images']:
            src = img_dir / img['file_name']
            if not src.exists():
                continue
            anns = [a for a in abi[img['id']] if a['bbox'][2] >= 10 and a['bbox'][3] >= 10]
            if not anns:
                continue
            W = img.get('width'); H = img.get('height')
            if not W or not H:
                with Image.open(src) as im:
                    W, H = im.size
            lines = []
            for a in anns:
                x, y, w, h = a['bbox']
                x = max(0.0, float(x)); y = max(0.0, float(y))
                w = min(float(w), W - x); h = min(float(h), H - y)
                if w < 5 or h < 5:
                    continue
                xc, yc = (x + w/2) / W, (y + h/2) / H
                wn, hn = w / W, h / H
                lines.append(f'{cat_map[a["category_id"]]} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
            if not lines:
                continue
            shutil.copy(src, out_img_dir / img['file_name'])
            with open(out_lbl_dir / (Path(img['file_name']).stem + '.txt'), 'w') as lf:
                lf.write('\n'.join(lines))
            n_written += 1
        print(f'  ✅ {sp:5s} → {n_written} images converted to YOLO format')

convert_coco_to_yolo(COCO_ROOT, ANNOTS_DIR, YOLO_ROOT, CLASS_NAMES)

data_yaml_path = os.path.join(YOLO_ROOT, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    yaml.dump({
        'path': YOLO_ROOT,
        'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
        'nc': NUM_CLASSES, 'names': CLASS_NAMES
    }, f)
print(f'\n✅ data.yaml written → {data_yaml_path}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Train YOLOv11n for 150 Epochs
# ═══════════════════════════════════════════════════════════
model = YOLO(MODEL_VARIANT)
print(f'✅ {MODEL_VARIANT} loaded — {sum(p.numel() for p in model.model.parameters()):,} parameters')

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=OUTPUT_DIR,
    name='yolo11n_run',
    patience=0,          # patience=0 -> never early-stop, always run the full EPOCHS
    verbose=True,
    plots=True,
)

run_dir = os.path.join(OUTPUT_DIR, 'yolo11n_run')
best_pt = os.path.join(run_dir, 'weights', 'best.pt')
print(f'\n✅ Training complete!  best.pt → {best_pt}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Loss Curves
# ═══════════════════════════════════════════════════════════
rdf = pd.read_csv(os.path.join(run_dir, 'results.csv'))
rdf.columns = [c.strip() for c in rdf.columns]

train_losses = (rdf['train/box_loss'] + rdf['train/cls_loss'] + rdf['train/dfl_loss']).tolist()
val_losses   = (rdf['val/box_loss']   + rdf['val/cls_loss']   + rdf['val/dfl_loss']).tolist()
epochs_ran = len(train_losses)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, epochs_ran+1), train_losses, label='Train Loss', color='#2196F3', linewidth=1.8)
ax.plot(range(1, epochs_ran+1), val_losses,   label='Val Loss',   color='#F44336', linewidth=1.8)
ax.set_title('YOLOv11n — Esophageal Cancer — Loss Curves', fontweight='bold', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (box + cls + dfl)')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlim(1, epochs_ran)
plt.tight_layout()
loss_path = os.path.join(OUTPUT_DIR, 'yolo11n_loss_curves.png')
plt.savefig(loss_path, dpi=150, bbox_inches='tight'); plt.show()
pd.DataFrame({'epoch': range(1, epochs_ran+1), 'train': train_losses, 'val': val_losses}).to_csv(
    os.path.join(OUTPUT_DIR, 'yolo11n_losses.csv'), index=False)
print(f'✅ Loss curves saved: {loss_path}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Evaluate on Test Set + Confusion Matrix
# ═══════════════════════════════════════════════════════════
best_model = YOLO(best_pt)
val_results = best_model.val(data=data_yaml_path, split='test', project=OUTPUT_DIR, name='yolo11n_test_eval')

nc = NUM_CLASSES
raw_matrix = val_results.confusion_matrix.matrix          # shape (nc+1, nc+1), rows=pred, cols=true, last=background
cm_array = raw_matrix[:nc, :nc].T.astype(int)              # -> (nc, nc), rows=true, cols=pred  (matches other notebooks)

print(f'✅ Evaluation complete')
print(f'Confusion matrix:\n{cm_array}')

cm_norm = cm_array.astype(float)
rs = cm_norm.sum(axis=1, keepdims=True); rs[rs==0]=1; cm_norm /= rs

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('YOLOv11n — Confusion Matrix — Esophageal Cancer', fontsize=13, fontweight='bold')
for ax, d, t, f in zip(axes, [cm_array, cm_norm], ['Raw Counts','Normalised'], ['d','.2f']):
    sns.heatmap(d, annot=True, fmt=f, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.8, annot_kws={'size':13,'weight':'bold'})
    ax.set_title(t, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'yolo11n_confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Confusion matrix saved: {cm_path}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Metrics Table
# ═══════════════════════════════════════════════════════════
def compute_metrics(cm, names):
    rows = []; total = cm.sum()
    for i, cls in enumerate(names[:cm.shape[0]]):
        TP=cm[i,i]; FP=cm[:,i].sum()-TP; FN=cm[i,:].sum()-TP; TN=total-TP-FP-FN
        P  = TP/(TP+FP) if TP+FP>0 else 0
        R  = TP/(TP+FN) if TP+FN>0 else 0
        F1 = 2*P*R/(P+R) if P+R>0 else 0
        Acc= (TP+TN)/total if total>0 else 0
        rows.append({'Class':cls,'TP':int(TP),'FP':int(FP),'FN':int(FN),'TN':int(TN),
                     'Precision':round(P,4),'Recall':round(R,4),
                     'F1-Score':round(F1,4),'Accuracy':round(Acc,4)})
    rows.append({'Class':'MACRO AVG','TP':'','FP':'','FN':'','TN':'',
                 'Precision':round(np.mean([r['Precision'] for r in rows]),4),
                 'Recall'   :round(np.mean([r['Recall']    for r in rows]),4),
                 'F1-Score' :round(np.mean([r['F1-Score']  for r in rows]),4),
                 'Accuracy' :round(np.mean([r['Accuracy']  for r in rows]),4)})
    return pd.DataFrame(rows)

mdf = compute_metrics(cm_array.astype(float), CLASS_NAMES)
print('='*65)
print('  YOLOv11n — PER CLASS METRICS')
print('='*65)
print(mdf.to_string(index=False))
print('='*65)

fig, ax = plt.subplots(figsize=(15, max(3, len(mdf)*0.65+1.2)))
ax.axis('off')
tbl = ax.table(cellText=mdf.values.tolist(), colLabels=list(mdf.columns), cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 1.8)
for j in range(len(mdf.columns)):
    tbl[0,j].set_facecolor('#1565C0')
    tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(mdf)):
    for j in range(len(mdf.columns)):
        tbl[i,j].set_facecolor('#F5F5F5' if i%2==0 else 'white')
last = len(mdf)
for j in range(len(mdf.columns)):
    tbl[last,j].set_facecolor('#BBDEFB')
    tbl[last,j].set_text_props(fontweight='bold')
ax.set_title('YOLOv11n — Precision / Recall / F1 / Accuracy', fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
tbl_path = os.path.join(OUTPUT_DIR, 'yolo11n_metrics_table.png')
plt.savefig(tbl_path, dpi=150, bbox_inches='tight'); plt.show()
mdf.to_csv(os.path.join(OUTPUT_DIR, 'yolo11n_metrics.csv'), index=False)
print(f'✅ Metrics table saved: {tbl_path}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Comparison Table: YOLOv11n vs your other trained models
# ═══════════════════════════════════════════════════════════
# 1) YOLOv11n's row comes straight from this notebook's own results (mdf, MACRO AVG row).
# 2) EDIT the dictionary below with the MACRO AVG Precision/Recall/F1/Accuracy from your
#    other trained models' *_metrics.csv MACRO AVG row.
# Point any *_METRICS_CSV path below to that model's CSV if it's available in this Kaggle
# session (e.g. attached as a Kaggle dataset/output) and it will be auto-loaded.

OTHER_MODEL_CSVS = {
    'Faster R-CNN'   : '/kaggle/input/fasterrcnn-results/fasterrcnn_metrics.csv',
    'RT-DETR'        : '/kaggle/input/rtdetr-results/rtdetr_metrics.csv',
    'SSD'            : '/kaggle/input/ssd-results/ssd_metrics.csv',
}

OTHER_MODEL_RESULTS = {name: {'Precision': None, 'Recall': None, 'F1-Score': None, 'Accuracy': None}
                        for name in OTHER_MODEL_CSVS}

for name, path in OTHER_MODEL_CSVS.items():
    if os.path.exists(path):
        fdf = pd.read_csv(path)
        frow = fdf[fdf['Class'] == 'MACRO AVG'].iloc[0]
        OTHER_MODEL_RESULTS[name] = {
            'Precision': float(frow['Precision']), 'Recall': float(frow['Recall']),
            'F1-Score': float(frow['F1-Score']), 'Accuracy': float(frow['Accuracy'])
        }
        print(f'✅ Loaded {name} metrics from {path}')
    else:
        print(f'⚠️  {name} CSV not found at {path} — fill OTHER_MODEL_RESULTS manually above.')

yolo_row = mdf[mdf['Class'] == 'MACRO AVG'].iloc[0]
comparison_rows = [{
    'Model': 'YOLOv11n', 'Precision': yolo_row['Precision'], 'Recall': yolo_row['Recall'],
    'F1-Score': yolo_row['F1-Score'], 'Accuracy': yolo_row['Accuracy']
}]
for name, vals in OTHER_MODEL_RESULTS.items():
    comparison_rows.append({'Model': name, **vals})

comp_df = pd.DataFrame(comparison_rows)
print('='*65)
print('  MODEL COMPARISON — MACRO AVG METRICS')
print('='*65)
print(comp_df.to_string(index=False))

plot_df = comp_df.dropna()
if len(plot_df) > 0:
    fig, ax = plt.subplots(figsize=(11, 6))
    x = np.arange(len(plot_df)); width = 0.2
    metrics_cols = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
    colors = ['#1565C0', '#43A047', '#FB8C00', '#8E24AA']
    for i, (m, c) in enumerate(zip(metrics_cols, colors)):
        ax.bar(x + i*width - 1.5*width, plot_df[m], width, label=m, color=c)
    ax.set_xticks(x); ax.set_xticklabels(plot_df['Model'], rotation=15)
    ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
    ax.set_title('Model Comparison — Esophageal Cancer Detection', fontweight='bold', fontsize=13)
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    comp_chart_path = os.path.join(OUTPUT_DIR, 'model_comparison_chart.png')
    plt.savefig(comp_chart_path, dpi=150, bbox_inches='tight'); plt.show()
    print(f'✅ Comparison chart saved: {comp_chart_path}')
else:
    print('⚠️  No other-model numbers filled in yet — skipping chart. Fill OTHER_MODEL_RESULTS and re-run this cell.')

comp_df.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=False)
print(f"✅ Comparison table saved: {os.path.join(OUTPUT_DIR, 'model_comparison.csv')}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Final Summary
# ═══════════════════════════════════════════════════════════
print('='*60)
print('  YOLOv11n — ALL OUTPUT FILES')
print('='*60)
for name, path in [
    ('best.pt',              best_pt),
    ('Loss Curves PNG',      os.path.join(OUTPUT_DIR,'yolo11n_loss_curves.png')),
    ('Loss Curves CSV',      os.path.join(OUTPUT_DIR,'yolo11n_losses.csv')),
    ('Confusion Matrix',     os.path.join(OUTPUT_DIR,'yolo11n_confusion_matrix.png')),
    ('Metrics Table PNG',    os.path.join(OUTPUT_DIR,'yolo11n_metrics_table.png')),
    ('Metrics CSV',          os.path.join(OUTPUT_DIR,'yolo11n_metrics.csv')),
    ('Comparison Chart PNG', os.path.join(OUTPUT_DIR,'model_comparison_chart.png')),
    ('Comparison CSV',       os.path.join(OUTPUT_DIR,'model_comparison.csv')),
]:
    mark = '✅' if os.path.exists(path) else '❌'
    print(f'  {mark}  {name:<22} → {path}')
print('\n🎉 YOLOv11n Training Complete!')
